In [1]:
import os
import sys
import time
from pathlib import Path

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.master("local[*]").appName("particiones").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

ruta = Path("../data/ventas_pentagrama_sim.csv").resolve().as_uri()
df = spark.read.csv(ruta, header=True, inferSchema=True)

print("Núcleos de tu computador:", os.cpu_count())
print("Particiones al leer el CSV:", df.rdd.getNumPartitions())

Núcleos de tu computador: 4
Particiones al leer el CSV: 1


In [2]:
df4 = df.repartition(4)
print("Particiones:", df4.rdd.getNumPartitions())

(
    df4.withColumn("particion", F.spark_partition_id())
    .groupBy("particion").count()
    .orderBy("particion")
    .show()
)

Particiones: 4
+---------+-----+
|particion|count|
+---------+-----+
|        0|   51|
|        1|   52|
|        2|   51|
|        3|   51|
+---------+-----+



In [3]:
(
    df.repartition(3, "vendedor")
    .withColumn("particion", F.spark_partition_id())
    .groupBy("particion", "vendedor").count()
    .orderBy("particion")
    .show()
)

+---------+------------+-----+
|particion|    vendedor|count|
+---------+------------+-----+
|        1|  Ana Torres|   65|
|        1|Luis Ramirez|   72|
|        2| Sofia Gomez|   68|
+---------+------------+-----+



In [4]:
df.groupBy("producto").agg(F.sum("monto").alias("total")).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[producto#19], functions=[sum(monto#22)])
   +- Exchange hashpartitioning(producto#19, 200), ENSURE_REQUIREMENTS, [plan_id=183]
      +- HashAggregate(keys=[producto#19], functions=[partial_sum(monto#22)])
         +- FileScan csv [producto#19,monto#22] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/ghote/Documents/Proyectos/pentagrama-sales-pipeline/dat..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<producto:string,monto:int>




In [5]:
df.filter(F.col("monto") > 300000).select("cliente", "monto").explain()

== Physical Plan ==
*(1) Filter (isnotnull(monto#22) AND (monto#22 > 300000))
+- FileScan csv [cliente#18,monto#22] Batched: false, DataFilters: [isnotnull(monto#22), (monto#22 > 300000)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/ghote/Documents/Proyectos/pentagrama-sales-pipeline/dat..., PartitionFilters: [], PushedFilters: [IsNotNull(monto), GreaterThan(monto,300000)], ReadSchema: struct<cliente:string,monto:int>




In [6]:
print(spark.sparkContext.uiWebUrl)
df4.groupBy("producto").count().collect()

http://192.168.20.127:4040


[Row(producto='Persiana enrollable', count=43),
 Row(producto='Cortina translucida', count=35),
 Row(producto='Persiana veneciana', count=43),
 Row(producto='Cortina blackout', count=48),
 Row(producto='Panel japones', count=36)]

In [7]:
spark.stop()

In [8]:
def medir(master):
    s = SparkSession.builder.master(master).appName("paralelo").getOrCreate()
    s.sparkContext.setLogLevel("ERROR")
    inicio = time.time()
    s.range(0, 40_000_000, 1, 16).select(
        ((F.col("id") % 1000) * F.sqrt(F.col("id"))).alias("x")
    ).agg(F.sum("x")).collect()
    segundos = time.time() - inicio
    s.stop()
    return segundos

t1 = medir("local[1]")
tn = medir("local[*]")
print(f"1 núcleo: {t1:.1f} s | todos los núcleos: {tn:.1f} s | mejora: {t1 / tn:.1f}x")

1 núcleo: 2.4 s | todos los núcleos: 1.4 s | mejora: 1.7x
